<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Test Environment for Generative AI classroom labs

This lab provides a test environment for the codes generated using the Generative AI classroom.

Follow the instructions below to set up this environment for further use.


# Setup


### Install required libraries

In case of a requirement of installing certain python libraries for use in your task, you may do so as shown below.


In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

### Dataset URL from the GenAI lab
Use the URL provided in the GenAI lab in the cell below. 


In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod2.csv"


### Downloading the dataset

Execute the following code to download the dataset in to the interface.

> Please note that this step is essential in JupyterLite. If you are using a downloaded version of this notebook and running it on JupyterLabs, then you can skip this step and directly use the URL in pandas.read_csv() function to read the dataset as a dataframe


In [3]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "dataset.csv")
file_name  = "dataset.csv"

---


# Test Environment


In [4]:
import pandas as pd
from pathlib import Path

def read_csv_with_header(file_path, sep=',', encoding='utf-8'):
    """
    Reads a CSV file into a pandas DataFrame.
    Assumes the first row contains the column headers.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"CSV file not found: {path}")

    df = pd.read_csv(path, sep=sep, encoding=encoding, header=0)
    return df

# Example usage:
df = read_csv_with_header("dataset.csv")
print(df.head())

<ipython-input-4-0ea4ffba5801>:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


   Unnamed: 0.1  Unnamed: 0 Manufacturer  Category  GPU  OS  CPU_core  \
0             0           0         Acer         4    2   1         5   
1             1           1         Dell         3    1   1         3   
2             2           2         Dell         3    1   1         7   
3             3           3         Dell         4    2   1         5   
4             4           4           HP         4    2   1         7   

   Screen_Size_inch  CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_pounds  \
0              14.0       0.551724       8             256        3.52800   
1              15.6       0.689655       4             256        4.85100   
2              15.6       0.931034       8             256        4.85100   
3              13.3       0.551724       8             128        2.69010   
4              15.6       0.620690       8             256        4.21155   

   Price Price-binned  Screen-Full_HD  Screen-IPS_panel  
0    978          Low               0   

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def train_and_evaluate_linear_regression(df, feature_col, target_col,
                                         test_size=0.2, random_state=42, verbose=True):
    """
    Train a simple linear regression model using a single feature column to predict a target column.
    Returns the trained model and evaluation metrics (MSE and R^2) on the test set.
    """
    if feature_col not in df.columns or target_col not in df.columns:
        raise ValueError("Specified feature or target column not found in the dataframe.")

    # Use only the two relevant columns and drop rows with NaNs
    data = df[[feature_col, target_col]].dropna()
    if data.shape[0] < 2:
        raise ValueError("Not enough valid data to train the model.")

    X = data[[feature_col]].values  # 2D array with shape (n_samples, 1)
    y = data[target_col].values      # 1D array with shape (n_samples,)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # Initialize and train the model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict on the test set
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if verbose:
        print(f"Linear Regression: feature='{feature_col}' -> target='{target_col}'")
        print(f"Coefficient (slope): {model.coef_[0]}")
        print(f"Intercept: {model.intercept_}")
        print(f"Test MSE: {mse:.6f}")
        print(f"Test R^2: {r2:.6f}")

    return {
        'model': model,
        'mse': mse,
        'r2': r2,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred
    }

# Example usage:
import numpy as np
df = pd.DataFrame({
     'feature': np.arange(100, dtype=float),
     'target': 2.5 * np.arange(100, dtype=float) + 10 + np.random.normal(scale=5.0, size=100)
 })
result = train_and_evaluate_linear_regression(df, 'feature', 'target', test_size=0.2, random_state=42)
print("MSE:", result['mse'], "R^2:", result['r2'])

Linear Regression: feature='feature' -> target='target'
Coefficient (slope): 2.4866756106341485
Intercept: 10.932811221077287
Test MSE: 27.042144
Test R^2: 0.994618
MSE: 27.042143713328517 R^2: 0.9946180858505425


In [10]:
def train_and_evaluate_linear_regression_multi(
    df,
    feature_cols,
    target_col,
    test_size=0.2,
    random_state=42,
    verbose=True
):
    """
    Train a linear regression model using multiple feature columns to predict a target column.
    Returns the trained model and evaluation metrics (MSE and R^2) on the test set.
    """
    # Validate inputs
    if not isinstance(feature_cols, (list, tuple)) or len(feature_cols) == 0:
        raise ValueError("feature_cols must be a non-empty list of column names.")
    for col in feature_cols + [target_col]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in the dataframe.")

    # Prepare data: drop rows with NaNs in involved columns
    data = df[feature_cols + [target_col]].dropna()
    if data.shape[0] < 2:
        raise ValueError("Not enough valid data to train the model.")

    X = data[feature_cols].values  # shape (n_samples, n_features)
    y = data[target_col].values     # shape (n_samples,)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Initialize and train the model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict on the test set
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if verbose:
        print(f"Linear Regression: features={feature_cols} -> target={target_col}")
        print("Coefficients:", dict(zip(feature_cols, model.coef_)))
        print("Intercept:", model.intercept_)
        print(f"Test MSE: {mse:.6f}")
        print(f"Test R^2: {r2:.6f}")

    return {
        'model': model,
        'mse': mse,
        'r2': r2,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred
    }

# Example usage (uncomment to run):
# import numpy as np
df = pd.DataFrame({
     'feature1': np.random.randn(200),
     'feature2': np.random.randn(200) * 5,
     'feature3': np.random.randn(200) + 2,
     'target': 3.0 * np.random.randn(200) + 1.5 * np.random.randn(200) + 5
 })
result = train_and_evaluate_linear_regression_multi(
     df,
    feature_cols=['feature1', 'feature2', 'feature3'],
     target_col='target',
     test_size=0.2,
     random_state=42
 )
print("MSE:", result['mse'], "R^2:", result['r2'])

Linear Regression: features=['feature1', 'feature2', 'feature3'] -> target=target
Coefficients: {'feature1': -0.1904628594660915, 'feature2': -0.036104219476237556, 'feature3': -0.09006231598920558}
Intercept: 5.422007338976554
Test MSE: 10.962156
Test R^2: -0.019286
MSE: 10.962156288034954 R^2: -0.019285832638570755


In [12]:
def train_and_evaluate_linear_regression_multi(
    df,
    feature_cols,
    target_col,
    test_size=0.2,
    random_state=42,
    verbose=True
):
    """
    Train a linear regression model using multiple feature columns to predict a target column.
    Returns the trained model and evaluation metrics (MSE and R^2) on the test set.
    """
    # Validate inputs
    if not isinstance(feature_cols, (list, tuple)) or len(feature_cols) == 0:
        raise ValueError("feature_cols must be a non-empty list of column names.")
    for col in feature_cols + [target_col]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in the dataframe.")

    # Prepare data: drop rows with NaNs in involved columns
    data = df[feature_cols + [target_col]].dropna()
    if data.shape[0] < 2:
        raise ValueError("Not enough valid data to train the model.")

    X = data[feature_cols].values  # shape (n_samples, n_features)
    y = data[target_col].values     # shape (n_samples,)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Initialize and train the model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict on the test set
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if verbose:
        print(f"Linear Regression: features={feature_cols} -> target={target_col}")
        print("Coefficients:", dict(zip(feature_cols, model.coef_)))
        print("Intercept:", model.intercept_)
        print(f"Test MSE: {mse:.6f}")
        print(f"Test R^2: {r2:.6f}")

    return {
        'model': model,
        'mse': mse,
        'r2': r2,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred
    }

# Example usage (uncomment to run):
# import numpy as np
df = pd.DataFrame({
     'feature1': np.random.randn(200),
     'feature2': np.random.randn(200) * 5,
     'feature3': np.random.randn(200) + 2,
     'target': 3.0 * np.random.randn(200) + 1.5 * np.random.randn(200) + 5
 })
result = train_and_evaluate_linear_regression_multi(
     df,
     feature_cols=['feature1', 'feature2', 'feature3'],
     target_col='target',
     test_size=0.2,
     random_state=42
 )
print("MSE:", result['mse'], "R^2:", result['r2'])

Linear Regression: features=['feature1', 'feature2', 'feature3'] -> target=target
Coefficients: {'feature1': 0.1177497202153119, 'feature2': 0.03436739861694548, 'feature3': 0.32018728718587536}
Intercept: 4.506367923202114
Test MSE: 7.910536
Test R^2: -0.032381
MSE: 7.910536341036861 R^2: -0.032380777089454416


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

def train_and_evaluate_pipeline_multifeatures(
    df,
    feature_cols,
    target_col,
    degree=2,
    test_size=0.2,
    random_state=42,
    verbose=True
):
    """
    Build a pipeline: PolynomialFeatures -> StandardScaler -> LinearRegression
    using multiple feature columns to predict a target column.
    Returns the trained pipeline and evaluation metrics (MSE, R^2) on the test set.
    """
    # Validate inputs
    if not isinstance(feature_cols, (list, tuple)) or len(feature_cols) == 0:
        raise ValueError("feature_cols must be a non-empty list of column names.")
    for col in feature_cols + [target_col]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in the dataframe.")

    # Prepare data: drop rows with NaNs in involved columns
    data = df[feature_cols + [target_col]].dropna()
    if data.shape[0] < 2:
        raise ValueError("Not enough valid data to train the model.")

    X = data[feature_cols].values  # shape (n_samples, n_features)
    y = data[target_col].values     # shape (n_samples,)

    # Define the pipeline: polynomial features, scaling, then linear regression
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Train the model
    pipeline.fit(X_train, y_train)

    # Predict on the test set
    y_pred = pipeline.predict(X_test)

    # Evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if verbose:
        print(f"Polynomial Regression (degree={degree}) with features={feature_cols} -> target={target_col}")
        print(f"Test MSE: {mse:.6f}")
        print(f"Test R^2: {r2:.6f}")

        # Optional: show some example coefficients (note: coefficients correspond to polynomial terms)
        # Access the trained model's coefficients via pipeline.named_steps
        poly_features = pipeline.named_steps['poly']
        model = pipeline.named_steps['model']
        # Get feature names for interpretability
        feature_names = poly_features.get_feature_names_out(input_features=feature_cols)
        coef = model.coef_
        coef_pairs = list(zip(feature_names, coef))
        coef_pairs_sorted = sorted(coef_pairs, key=lambda kv: abs(kv[1]), reverse=True)[:5]
        print("Top terms by |coef|:", coef_pairs_sorted)

    return {
        'pipeline': pipeline,
        'mse': mse,
        'r2': r2,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred
    }

# Example usage (uncomment and replace with your data):
# import numpy as np
df = pd.DataFrame({
     'feature1': np.random.randn(200),
     'feature2': np.random.randn(200) * 2.0,
     'feature3': np.random.randn(200) + 1.0,
     'target': None  # replace with your actual target column
 })
# # Create a synthetic target for demonstration, e.g. a nonlinear combo
df['target'] = 1.5*df['feature1'] - 0.8*df['feature2'] + 0.5*df['feature3']**2 + np.random.randn(200)*0.5

result = train_and_evaluate_pipeline_multifeatures(
     df,
     feature_cols=['feature1', 'feature2', 'feature3'],
     target_col='target',
     degree=3,          # degrees can be 2, 3, 5, etc.
 test_size=0.2,
 random_state=42
 )
print("MSE:", result['mse'], "R^2:", result['r2'])

Polynomial Regression (degree=3) with features=['feature1', 'feature2', 'feature3'] -> target=target
Test MSE: 0.420152
Test R^2: 0.902823
Top terms by |coef|: [('feature1', 1.6730269704037468), ('feature2', -1.5831279868869539), ('feature3^2', 1.056832587457402), ('feature1 feature3', -0.23089285944674953), ('feature2 feature3', 0.22177524698007003)]
MSE: 0.42015249560477297 R^2: 0.9028230457798048


In [9]:
def normalize_cpu_frequency_in_place(df: pd.DataFrame) -> None:
    """
    Normalize the 'CPU_frequency' column by its maximum value.
    - Converts the column to numeric (coercing non-numeric to NaN)
    - Divides by the maximum value (in-place)
    - If the column is missing or the max is NaN or <= 0, no change is made
    """
    if 'CPU_frequency' not in df.columns:
        return

    # Ensure numeric for proper max calculation
    df['CPU_frequency'] = pd.to_numeric(df['CPU_frequency'], errors='coerce')

    max_val = df['CPU_frequency'].max()
    if pd.isna(max_val) or max_val <= 0:
        return  # nothing to scale or invalid max

    df['CPU_frequency'] = df['CPU_frequency'] / max_val

# Example usage:
df = pd.read_csv("dataset.csv")
normalize_cpu_frequency_in_place(df)
print(df['CPU_frequency'].head())

0    0.551724
1    0.689655
2    0.931034
3    0.551724
4    0.620690
Name: CPU_frequency, dtype: float64


In [10]:
import re

def _sanitize_value(val: object) -> str:
    s = str(val).strip()
    s = s.replace(' ', '_').replace('-', '_')
    # keep only alphanumeric and underscore
    s = re.sub(r'[^A-Za-z0-9_]', '', s)
    return s

def convert_screen_to_indicators(df: pd.DataFrame):
    """
    1) Create indicator columns for each unique value in df['Screen'].
       Columns are named Screen_<unique_value>, with simple sanitization.
    2) Append these indicator columns to the original df.
    3) Drop the original 'Screen' column.

    Returns:
      df_out: DataFrame with the original columns (except 'Screen') plus indicators
      df1: DataFrame of the indicator columns (saved as df1)
    """
    df_out = df.copy()
    df1 = pd.DataFrame(index=df_out.index)

    if 'Screen' in df_out.columns:
        uniques = df_out['Screen'].dropna().unique()
        for val in uniques:
            col_name = f"Screen_{_sanitize_value(val)}"
            df1[col_name] = (df_out['Screen'] == val).astype(int)

        # Append indicators to the original DataFrame
        df_out = pd.concat([df_out, df1], axis=1)
        # Drop the original 'Screen' column
        df_out = df_out.drop(columns=['Screen'])

    return df_out, df1

# Example usage:
df_out, df1 = convert_screen_to_indicators(df)
print(df1.head())
print(df_out.head())

   Screen_IPS_Panel  Screen_Full_HD
0                 1               0
1                 0               1
2                 0               1
3                 1               0
4                 0               1
   Unnamed: 0 Manufacturer  Category  GPU  OS  CPU_core  Screen_Size_cm  \
0           0         Acer         4    2   1         5          35.560   
1           1         Dell         3    1   1         3          39.624   
2           2         Dell         3    1   1         7          39.624   
3           3         Dell         4    2   1         5          33.782   
4           4           HP         4    2   1         7          39.624   

   CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  Screen_IPS_Panel  \
0       0.551724       8             256       1.60    978                 1   
1       0.689655       4             256       2.20    634                 0   
2       0.931034       8             256       2.20    946                 0   
3       0.551

In [11]:
from typing import Optional
import requests
import json

def convert_price_usd_to_eur(
    df: pd.DataFrame,
    rate: Optional[float] = None,
    column: str = "Price",
    new_column: Optional[str] = None,
    fetch_rate: bool = False,
    api_url: str = "https://api.exchangerate.host/latest"
) -> pd.DataFrame:
    """
    Convert USD prices to EUR.

    - Coerce the source column to numeric (non-numeric -> NaN).
    - Use a provided rate or fetch the rate from the API if fetch_rate is True.
    - Write results to a new column if new_column is provided; otherwise overwrite the source column.
    - Modify the DataFrame in place and return it for chaining.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")

    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame")

    # Coerce to numeric (non-numeric -> NaN)
    usd_values = pd.to_numeric(df[column], errors="coerce")

    # Determine rate
    if rate is not None:
        if rate <= 0:
            raise ValueError("rate must be positive")
        usd_to_eur = float(rate)
    elif fetch_rate:
        try:
            resp = requests.get(api_url, params={"base": "USD", "symbols": "EUR"}, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # Handle common error shapes
            if isinstance(data, dict) and data.get("success") is False:
                raise ValueError(
                    "API request failed. Response: " + json.dumps(data)
                )

            rates = data.get("rates") or {}
            eur_rate = rates.get("EUR")

            # Fallback: case-insensitive key match
            if eur_rate is None:
                for k, v in rates.items():
                    if str(k).upper() == "EUR":
                        eur_rate = v
                        break

            if eur_rate is None:
                raise ValueError(
                    "EUR rate not found in API response. Response: "
                    + json.dumps(data)[:300]
                )

            usd_to_eur = float(eur_rate)
            if usd_to_eur <= 0:
                raise ValueError("Fetched rate must be positive.")
        except requests.RequestException as e:
            raise ConnectionError(f"Failed to fetch USD->EUR rate: {e}")
        except ValueError:
            raise
    else:
        raise ValueError("Provide a rate or set fetch_rate=True to fetch the rate from the API.")

    eur_values = usd_values * usd_to_eur

    if new_column:
        df[new_column] = eur_values
    else:
        df[column] = eur_values

    return df

# Example usage:
# Correct DataFrame creation
df = pd.DataFrame({"Price": ["10", 20, None, "abc"]})

# Overwrite the Price column with a provided rate
df_out = convert_price_usd_to_eur(df, rate=0.92)

print(df_out)

   Price
0    9.2
1   18.4
2    NaN
3    NaN


In [12]:
import numpy as np
from typing import Optional

def min_max_normalize_cpu_frequency(
    df: pd.DataFrame,
    column: str = "CPU_frequency",
    new_column: Optional[str] = None
) -> pd.DataFrame:
    """
    Apply min-max normalization to the CPU_frequency column.

    - Non-numeric values are coerced to NaN.
    - Min/Max are computed from non-NaN values.
    - If max != min, normalize: (x - min) / (max - min).
    - If max == min or min/max is NaN, set normalized value to 0.0 for non-NaN entries.
    - NaNs remain NaN in the result.
    - If new_column is provided, write to that column; otherwise overwrite the source column.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")

    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame")

    # Coerce to numeric
    vals = pd.to_numeric(df[column], errors="coerce")

    minv = vals.min()
    maxv = vals.max()

    # Compute denominator if possible
    denom = None
    if pd.notna(minv) and pd.notna(maxv):
        denom = maxv - minv

    # Normalize
    if denom is not None and denom != 0:
        norm = (vals - minv) / denom
        norm = norm.astype(float)
    else:
        # max == min or invalid min/max: set 0.0 for non-NaN entries
        norm = pd.Series(np.nan, index=vals.index, dtype=float)
        non_nan_idx = vals.notna()
        norm[non_nan_idx] = 0.0

    # Write result
    if new_column:
        df[new_column] = norm
    else:
        df[column] = norm

    return df

# Example usage:
df = pd.DataFrame({"CPU_frequency": ["2.5", 3.0, None, "not a number", 1.5]})
df_norm = min_max_normalize_cpu_frequency(df, new_column="CPU_frequency_norm")
print(df_norm)

  CPU_frequency  CPU_frequency_norm
0           2.5            0.666667
1           3.0            1.000000
2          None                 NaN
3  not a number                 NaN
4           1.5            0.000000


In [14]:
print(df.columns.tolist())

['CPU_frequency', 'CPU_frequency_norm']


In [13]:
import seaborn as sns
import matplotlib.pyplot as plt

# Optional: improve visuals
sns.set(style="whitegrid")

# Load your data into a DataFrame named df
# Example:
# df = pd.read_csv('path_to_your_dataset.csv')
# If you already have df in memory, skip the above and proceed.

# Ensure required columns exist
required = [
    'Price',
    'CPU_frequency',
    'Screen_Size_inch',
    'Weight_pounds',
    'Category',
    'GPU',
    'OS',
    'CPU_core',
    'RAM_GB',
    'Storage_GB_SSD'
]
missing = [c for c in required if c not in globals().get('df', {}) and 'df' not in globals()]
# (Optional) you can add a runtime check or simply assume df exists.

# Helper: prepare a DataFrame for a box plot against Price
def prepare_boxplot_df(df, attr, max_unique=12):
    sub = df[[attr, 'Price']].copy()
    if pd.api.types.is_numeric_dtype(sub[attr]):
        # If too many unique numeric values, bin into ~max_unique categories
        if sub[attr].nunique() > max_unique:
            sub[attr] = pd.cut(sub[attr], bins=max_unique).astype(str)
        else:
            sub[attr] = sub[attr].astype(str)
    else:
        sub[attr] = sub[attr].astype(str)
    return sub

# 1) Regression plots: Price vs each numeric attribute
plt.figure(figsize=(6, 4))
sns.regplot(x='CPU_frequency', y='Price', data=df,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.xlabel('CPU_frequency')
plt.ylabel('Price')
plt.title('Price vs CPU_frequency')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.regplot(x='Screen_Size_inch', y='Price', data=df,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.xlabel('Screen_Size_inch')
plt.ylabel('Price')
plt.title('Price vs Screen_Size_inch')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.regplot(x='Weight_pounds', y='Price', data=df,
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.xlabel('Weight_pounds')
plt.ylabel('Price')
plt.title('Price vs Weight_pounds')
plt.tight_layout()
plt.show()

# 2) Box plots: Price by each categorical attribute
# Attributes to plot against Price (convert numeric attrs to categorical where needed)
attributes_for_boxplot = ['Category', 'GPU', 'OS', 'CPU_core', 'RAM_GB', 'Storage_GB_SSD']

for attr in attributes_for_boxplot:
    # Prepare data for box plot
    box_df = prepare_boxplot_df(df, attr, max_unique=12)
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=attr, y='Price', data=box_df)
    plt.xticks(rotation=45)
    plt.xlabel(attr)
    plt.ylabel('Price')
    plt.title(f'Price by {attr}')
    plt.tight_layout()
    plt.show()

<Figure size 600x400 with 0 Axes>

<class 'KeyError'>: 'Price'

## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-12-10|0.1|Abhishek Gagneja|Initial Draft created|


Copyright © 2023 IBM Corporation. All rights reserved.
